# Chapter 3.3.1 - A simple self-attention mechanism without trainable weights

In [1]:
import torch

##### lets take a sentence as input

##### Your journey starts with one step

In [3]:
inputs=torch.tensor([
[0.43, 0.15, 0.89], #embedding vector of token "your" 
[0.55, 0.87, 0.66], #embedding vector of token "journey" 
[0.57, 0.85, 0.64], #embedding vector of token "starts" 
[0.22, 0.58, 0.33], #embedding vector of token "with" 
[0.77, 0.25, 0.10], #embedding vector of token "one " 
[0.05, 0.80, 0.55]] #embedding vector of token "step" 
)

##### each row above is the embedding of one token. so all together for 6 tokens we have 6 embedding vectors of 3 dim

In [4]:
print(inputs[1]) 

tensor([0.5500, 0.8700, 0.6600])


In [5]:
inputs.shape

torch.Size([6, 3])

##### what we need to do now is we need to calculate so called attention score, and how we do it is by taking one token's embedding vector at a time as input and doing the dot product with itself and all the other token's embedding vectors.

##### to understand how this works, lets take the first token embedding vector as input and calculate its  dot product with all

In [54]:
# select the first token's embedding as the query vector.
query = inputs[0]

# create an empty tensor to store one attention score
# for each input token.
# inputs.shape[0] = 6, so attn_scores will have 6 values.
attn_scores = torch.empty(inputs.shape[0])

# loop through all input token embeddings.
for i in range(inputs.shape[0]):

    # compute the dot product between the query embedding
    # and the current token's embedding (inputs[i]).
    # each dot product returns a single value. Since there
    # are 6 input tokens, we compute 6 dot products,
    # resulting in 6 attention scores.
    # each score represents the similarity between the
    # query embedding and one input embedding.
    attn_scores[i] = torch.dot(query, inputs[i])

# print the attention scores
print(attn_scores)

tensor([0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310])


In [55]:
##### sample example of how dot product works
query=inputs[0]
for i in range(inputs.shape[0]):
    print("query=" ,query)
    print("input=" ,inputs[i])
    multiply=query* inputs[i]
    print("element wise multiplication: \n ",multiply)
    final_value=torch.sum((multiply))
    print("output_value after addition of all = ",final_value)
    print("*****************************************************")

query= tensor([0.4300, 0.1500, 0.8900])
input= tensor([0.4300, 0.1500, 0.8900])
element wise multiplication: 
  tensor([0.1849, 0.0225, 0.7921])
output_value after addition of all =  tensor(0.9995)
*****************************************************
query= tensor([0.4300, 0.1500, 0.8900])
input= tensor([0.5500, 0.8700, 0.6600])
element wise multiplication: 
  tensor([0.2365, 0.1305, 0.5874])
output_value after addition of all =  tensor(0.9544)
*****************************************************
query= tensor([0.4300, 0.1500, 0.8900])
input= tensor([0.5700, 0.8500, 0.6400])
element wise multiplication: 
  tensor([0.2451, 0.1275, 0.5696])
output_value after addition of all =  tensor(0.9422)
*****************************************************
query= tensor([0.4300, 0.1500, 0.8900])
input= tensor([0.2200, 0.5800, 0.3300])
element wise multiplication: 
  tensor([0.0946, 0.0870, 0.2937])
output_value after addition of all =  tensor(0.4753)
**********************************************

##### we can see that the output value matces with tensor([0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310])

##### once we obtain the attention score, the next step is to normalize the attention scores to obtain attention weights  that sum up to 1. This normalization is a convention that is useful for interpretation and for maintaining training stability in an LLM

#### if we calculate the sum of attn score right now, its past 1.

In [61]:
sum_of_attn_scores=attn_scores.sum()
print(sum_of_attn_scores) 

tensor(4.4600)


##### to normalize it so that its sum becomes one, we take the attention score and divide it by its sum

In [62]:
attn_weights_temp= attn_scores/ sum_of_attn_scores
print(attn_weights_temp)

tensor([0.2241, 0.2140, 0.2113, 0.1066, 0.1026, 0.1415])


##### after normalizing if we calculate the sum of new normalized weights,it should be 1.

In [64]:
print(attn_weights_temp.sum())

tensor(1.)


##### in practice, it's more common and advisable to use the softmax function for normalization. this approach is better at managing extreme values and offers more favorable gradient properties during training. below is a basic implementation of the softmax function for normalizing the attention scores:

In [67]:
print(attn_scores.sum())

tensor(4.4600)


In [70]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights= softmax_naive(attn_scores)
print(attn_weights)
print(attn_weights.sum())

tensor([0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452])
tensor(1.0000)


##### as the output shows, the softmax function also meets the objective and normalizes the attention weights such that they sum to 1:

In addition, the softmax function ensures that the attention weights are always positive.
This makes the output interpretable as probabilities or relative importance, where higher
weights indicate greater importance.
Note that this naive softmax implementation (softmax_naive) may encounter numerical
instability problems, such as overflow and underflow, when dealing with large or small input
values. Therefore, in practice, it's advisable to use the PyTorch implementation of softmax,
which has been extensively optimized for performance:


In [71]:
print(attn_scores)

tensor([0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310])


In [72]:
attn_weights= torch.softmax(attn_scores,dim=0)
print(attn_weights)
print(attn_weights.sum())

tensor([0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452])
tensor(1.0000)


##### we can see that it yields the same results as our previous softmax_naive function, but this implemenationis highly optimizede for performance

##### now that computed the normalized attention weights, we are ready for the final step ie calculating the context vector z by multiplying the embedded input tokens with the corresponding attention weights and then summing the resulting vectors.

In [73]:
query=inputs[0]
print(query.shape)

torch.Size([3])


In [86]:
print(inputs[0])

tensor([0.4300, 0.1500, 0.8900])


In [87]:
query=inputs[0]
#create a zeero vector with same shape as query embedding to store the context values.
contxt_vec= torch.zeros(query.shape)
#compute context vectors and add them to the contxt_vec.
for i in range(inputs.shape[0]):
    contxt_vec= contxt_vec+ inputs[i]* attn_weights[i] 
print(contxt_vec)

tensor([0.4421, 0.5931, 0.5790])


In [88]:
###### In the next section, we will generalize this procedure for computing context vectors to calculate all context vectors simultaneously